# Python reference 4: collections, heapq and bisect

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-04-collections-heapq-bisect.ipynb)

`Counter`, `defaultdict`, `OrderedDict`, `deque`, `namedtuple` and `ChainMap`; heaps with `heapq` (top-K, merging, priority queues); and binary search with `bisect`. Practise it in [Python practice 4: collections, heapq and bisect](../practice/python-practice-04-collections-heapq-bisect.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [collections](#collections) | [`Counter`](#counter), [`defaultdict`](#defaultdict), [`OrderedDict`](#ordereddict), [`deque`](#deque), [`namedtuple`](#namedtuple), [`ChainMap`](#chainmap) |
| [heapq](#heapq) | [Heaps with `heapq`](#heaps-with-heapq), [Top-K over a stream](#top-k-over-a-stream) |
| [bisect](#bisect) | [Binary search with `bisect`](#binary-search-with-bisect) |

## collections

### `Counter`

A dict subclass that counts hashable items. Missing keys read as 0 (without being inserted).

| member | returns |
| --- | --- |
| `Counter(iterable)` / `Counter(mapping)` / `Counter(a=1)` | a counter |
| `c.most_common(n=None)` | `(item, count)` pairs, highest first; ties keep first-seen order |
| `c.update(iterable)` / `c.subtract(iterable)` | add / subtract counts in place |
| `c.total()` (3.10+) | sum of counts |
| `c.elements()` | each item repeated by its count |
| `a + b`, `a - b`, `a & b`, `a \| b` | add, subtract (drops ≤ 0), min, max of counts |

**Notes:** `most_common` breaks ties by insertion order, not alphabetically; sort explicitly when ties must be
deterministic by value.

In [ ]:
from collections import ChainMap, Counter, OrderedDict, defaultdict, deque, namedtuple

words = "the cat and the hat and the bat".split()
c = Counter(words)
print(c.most_common(2), c["dog"], "dog" in c, c.total())
c.update(["cat", "cat"]); c.subtract(["the"])
print(c, sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))[:3])
print(Counter(a=3, b=1) - Counter(a=1, b=2), Counter(a=3) & Counter(a=1, b=2), list(Counter(x=2).elements()))

### `defaultdict`

`defaultdict(factory)` calls `factory()` to create a value the first time a missing key is **read**. Common factories:
`list`, `set`, `int`, `dict`, or `lambda: defaultdict(int)` for nested counters.

**Notes:** merely reading `d[key]` inserts the key, which can grow the dict unexpectedly (and makes `key in d` true
afterwards). Use `d.get(key)` to look without inserting; convert to `dict` before returning it from an API.

In [ ]:
by_region = defaultdict(list)
for region, order_id in [("north", 1), ("south", 2), ("north", 3)]:
    by_region[region].append(order_id)
print(dict(by_region))
nested = defaultdict(lambda: defaultdict(int))
for region, day, amount in [("north", "d1", 10), ("north", "d1", 5), ("south", "d2", 7)]:
    nested[region][day] += amount
print({k: dict(v) for k, v in nested.items()})
counts = defaultdict(int)
_ = counts["ghost"]
print("reading inserted a key:", dict(counts))

### `OrderedDict`

Plain dicts keep insertion order, so `OrderedDict` is now for its extra operations: `move_to_end(key, last=True)`,
`popitem(last=True)` (pop from either end), and order-sensitive equality. It is the classic building block of an LRU
cache.

In [ ]:
od = OrderedDict.fromkeys("abc")
od.move_to_end("a")
print(list(od), od.popitem(last=False), OrderedDict(a=1, b=2) == OrderedDict(b=2, a=1), dict(a=1, b=2) == dict(b=2, a=1))

### `deque`

A double-ended queue with O(1) appends and pops at **both** ends; lists are O(n) at the front.

| method | effect |
| --- | --- |
| `append(x)`, `appendleft(x)` | add at the right / left |
| `pop()`, `popleft()` | remove from the right / left |
| `extend(it)`, `extendleft(it)` | add many (`extendleft` reverses them) |
| `rotate(n)` | rotate right by `n` (left if negative) |
| `deque(it, maxlen=k)` | bounded: adding beyond `k` drops from the other end |

**Use it for:** queues (BFS), sliding windows, keeping the last N lines of a log.

In [ ]:
q = deque([1, 2, 3])
q.appendleft(0); q.append(4)
print(q.popleft(), q.pop(), q)
q.rotate(1); print(q)
last3 = deque(maxlen=3)
for line in ["a", "b", "c", "d", "e"]:
    last3.append(line)
print(list(last3))

### `namedtuple`

`namedtuple("Name", ["field", ...], defaults=...)` creates a lightweight immutable record class with named fields that
is still a tuple (unpackable, hashable, memory-light).

| member | returns |
| --- | --- |
| `p.field` / `p[0]` | field by name or position |
| `p._asdict()` | a dict |
| `p._replace(field=value)` | a new record with changed fields |
| `Name._fields` | field names |
| `Name._make(iterable)` | build from a sequence (a CSV row) |

**Notes:** for records with defaults, type hints and methods, prefer `@dataclass(frozen=True)`
([classes](python-reference-07-classes-and-dataclasses.ipynb)); `typing.NamedTuple` is the typed form.

In [ ]:
Order = namedtuple("Order", ["order_id", "status", "amount"], defaults=[0.0])
o = Order._make(["7", "paid", "310.0"])
print(o, o.status, o[0], o._asdict(), o._replace(status="shipped"), Order._fields, Order(1, "placed"))
order_id, status, amount = o
print(order_id, status, amount, hash(o) == hash(Order("7", "paid", "310.0")))

### `ChainMap`

`ChainMap(first, second, ...)` looks keys up in each mapping in turn, without copying them. Writes go to the first
mapping. It models layered configuration: command-line overrides, then environment, then defaults.

In [ ]:
defaults = {"retries": 3, "timeout": 30, "debug": False}
env = {"timeout": 60}
cli = {"debug": True}
config = ChainMap(cli, env, defaults)
print(config["timeout"], config["debug"], config["retries"], dict(config))
config["retries"] = 5
print(cli, defaults["retries"])

## heapq

### Heaps with `heapq`

`heapq` keeps a plain list as a **min-heap**: `heap[0]` is always the smallest item.

| function | effect | cost |
| --- | --- | --- |
| `heapify(list)` | rearrange a list into a heap in place | O(n) |
| `heappush(heap, x)` / `heappop(heap)` | add / remove the smallest | O(log n) |
| `heappushpop(heap, x)` | push then pop (faster than both) | O(log n) |
| `heapreplace(heap, x)` | pop then push | O(log n) |
| `nlargest(n, it, key=None)` / `nsmallest` | the n largest / smallest | O(N log n) |
| `merge(*sorted_iterables, key=None, reverse=False)` | lazily merge already-sorted inputs | O(N log k) |

**Notes:** there is no max-heap; push negated keys `(-score, item)`. Tuples compare element by element, so add a
counter as a tie-breaker when items themselves are not comparable: `(priority, next(counter), item)`.

In [ ]:
import heapq
from itertools import count

nums = [5, 1, 8, 3, 9, 2]
heap = nums[:]; heapq.heapify(heap)
print(heap[0], [heapq.heappop(heap) for _ in range(3)], heapq.nlargest(2, nums), heapq.nsmallest(2, nums))
orders = [{"id": 1, "amount": 120}, {"id": 9, "amount": 1180}, {"id": 7, "amount": 310}]
print([o["id"] for o in heapq.nlargest(2, orders, key=lambda o: o["amount"])])
print(list(heapq.merge([1, 4, 9], [2, 3, 10], [5])))
tasks, tie = [], count()
for priority, task in [(2, {"name": "load"}), (1, {"name": "extract"}), (2, {"name": "notify"})]:
    heapq.heappush(tasks, (priority, next(tie), task))
print([heapq.heappop(tasks)[2]["name"] for _ in range(3)])

### Top-K over a stream

To keep the K largest items of a long stream in O(n log K) time and O(K) memory, keep a **min-heap of size K**: push
until full, then `heappushpop` each new item, which drops the current smallest.

In [ ]:
def top_k(stream, k):
    heap = []
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        else:
            heapq.heappushpop(heap, x)
    return sorted(heap, reverse=True)

print(top_k(iter([5, 1, 8, 3, 9, 2, 7]), 3))

## bisect

### Binary search with `bisect`

`bisect` works on a **sorted** list in O(log n).

| function | returns / does |
| --- | --- |
| `bisect_left(a, x, lo=0, hi=len(a), key=None)` | first index where `x` could be inserted (before equal items) |
| `bisect_right(a, x, ...)` / `bisect` | index after any equal items |
| `insort_left` / `insort_right` / `insort` | insert keeping the list sorted (the insert itself is O(n)) |
| `key=` (3.10+) | compare by a key of the list items |

**Use it for:** bucketing values by thresholds (tax brackets, tiers), counting items in a range
(`bisect_right(a, hi) - bisect_left(a, lo)`), and keeping a small sorted list.

In [ ]:
import bisect

thresholds, tiers = [100, 500, 1000], ["small", "medium", "large", "enterprise"]
print([tiers[bisect.bisect_right(thresholds, amount)] for amount in [30, 100, 499, 500, 1180]])
scores = [10, 20, 20, 20, 30]
print(bisect.bisect_left(scores, 20), bisect.bisect_right(scores, 20), bisect.bisect_right(scores, 25) - bisect.bisect_left(scores, 15))
bisect.insort(scores, 25); print(scores)
events = [(1, "a"), (5, "b"), (9, "c")]
print(bisect.bisect_left(events, 6, key=lambda e: e[0]))

**Next:** practise these entries in [Python practice 4: collections, heapq and bisect](../practice/python-practice-04-collections-heapq-bisect.ipynb).